# Load deps

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

# Create OpenAI client

Check that the OpenAI client works

In [ ]:
openai_client = OpenAI()

# Plain LLMs lack our data

First, let's define a function to talk to the LLM:

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

This is our black box - text goes in, text comes out.

Let's test it:

In [ ]:
llm("Hey, what's up?")

Ask it a course-specific question:

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

This is different from a question like "how do I cook salmon?" - the LLM knows the answer because cooking salmon is common knowledge. But our courses are not in the training data.

# Adding context manually

More context can fix this. The FAQ website has questions and answers about our courses.

Copy some of that content into the prompt:

In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
answer = llm(prompt)
print(answer)

# Retrieval plus generation

RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. That search step is what gives the LLM the context it needs to answer correctly.

What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

In code, it looks like this:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```
That's the entire architecture. It comes down to three components.

The pieces are search, the prompt, and the LLM:
-   search
-   prompt
-   LLM


The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, so search quality matters a lot for RAG.

The database and the LLM can be anything. Because each piece is independent, RAG stays flexible.
- for now, we use minsearch and then sqlitesearch for search, and OpenAI for the LLM
- To use Anthropic instead of OpenAI, you swap the LLM call.
- To use Elasticsearch instead of minsearch, you swap the search call. Nothing else changes.

## The Course FAQ Dataset

Before we build the RAG pipeline, let's get familiar with the data we'll use as our knowledge base.

The FAQ data is available as JSON from the DataTalks.Club website.

Let's fetch it:

In [ ]:
import requests
from pprint import pprint

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
pprint(courses_raw)

Let's fetch all the FAQ documents from all courses:

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

In [ ]:
print(len(documents))
pprint(documents[3])

## Using FAQ data

In the RAG pipeline, this dataset is our knowledge base:

1.  We index all the documents (the search step)
2.  When a student asks a question, we search the index
3.  The search returns the most relevant FAQ entries
4.  We give those entries to the LLM as context
5.  The LLM generates an answer based on the context

- The `question` and `answer` fields contain the text we'll search through.
- The `course` field lets us filter by course. For example, if a student asks about the data engineering course, we skip results from the ML course.
- The `section` field helps with ranking - knowing which part of the course a question belongs to is useful context.

### A note on data preparation

- In our case, the data is already prepared in a convenient JSON format. So we don't need to do much to get it ready.
- Don't let that fool you. In reality, data preparation is often the most time-consuming part of building a RAG system. You may need to scrape websites, parse PDFs, and clean and chunk documents. That work isn't visible here.

## Search Engine

At its core, every search engine does the same thing. It takes a query, scores every document for similarity, and returns the top results.

Formally, there is a similarity function:

```python
score = sim(query, document)
```

For each document in the database, you compute this score. Then you rank all documents by score and return the top N. What makes a search engine different from another search engine is what `sim` actually computes.

-   text/lexical search: `sim` counts how many words the query and the document share. It looks at the surface form, the actual words, and matches them exactly.
-   vector/semantic search: `sim` compares the meaning of the query and the document. Same function, different similarity measure.

Consider these two questions:
-   "Can I still join the course after the start date?"
-   "Is it possible to enroll late?"

They mean the same thing, but share almost no keywords. "Join" is not "enroll", "course" is absent, "start date" is not "late". A text search engine would struggle to match them, because it only sees words.

**Searching matters** because we have around 1100 documents. Sending all of them to the LLM would be expensive and slow. The model would get confused with too much data. Search finds the most relevant documents to send instead.

### Indexing with minsearch

There are many search libraries you can use:
- Apache Lucene,
- Elasticsearch,
- Solr,
- and others.
But these are somewhat heavy. For example, to run Elasticsearch, you need to start a Docker container.

[minsearch](https://github.com/alexeygrigorev/minsearch) is a simple in-memory search engine. It's lightweight, so it runs anywhere Python runs, including Google Colab where you can't start a Docker container. It's a toy implementation, not production ready, but it illustrates how search engines work and it gives good results.

[Alexey](https://github.com/alexeygrigorev) wrote this library and maintains it. It started as a single Python file in the first edition of LLM Zoomcamp. He wrote it as part of the [Build a Search Engine](https://www.youtube.com/watch?v=nMrGK5QgPVE) workshop (see the [code](https://github.com/alexeygrigorev/build-your-own-search-engine)).

- The concepts in minsearch are the same as in Elasticsearch (which comes from Lucene): text fields, keyword fields, boosting, filtering. So what you learn here transfers directly.
- We'll index the `question`, `section`, and `answer` fields as text (they'll be tokenized and ranked), and the `course` field as a keyword (for filtering).
- The index tokenizes text fields and treats keyword fields as exact strings.
- Text fields are the fields you search through. When you type a query, the search engine looks at these fields and tokenizes them. It splits text into words, lowercases them, removes stop words, and ranks the results by relevance. So `question`, `section`, and `answer` are text fields.
- Keyword fields are for exact matching. Think of a SQL query like `SELECT * FROM index WHERE course = 'data-engineering-zoomcamp'`. The results must come from the specified course, no matter what ranking or boosting you do for text fields. You use keyword fields to restrict the search space to a particular subset. In our case, we have four courses. Say you're taking the LLM course and ask a question. You don't want answers from the MLOps or machine learning courses mixed in.

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

### Trying a search

Let's try a search with the question we used before:
- We use `boost_dict` to give the `question` field more weight (2.0) and `section` less weight (0.5).
    - All fields have a default boost of 1.0
    - Query words appearing in the question field is a stronger signal than them appearing in the section name.
- We use `filter_dict` to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
pprint([doc["question"] for doc in search_results])

In [ ]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)
pprint([doc["question"] for doc in results])

### Our first search function

Let's wrap the search in a `search` function - the first component of our RAG pipeline:

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
search_results = search(question)
pprint([doc["question"] for doc in search_results])

## Building the Prompt

The LLM doesn't see our documents unless we pass them in. So we need to build a prompt that includes the user's question and the search
results.

When we build AI systems, we usually split the prompt into two parts:
- Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.
- User prompt: this changes with every request. It carries the actual question and the retrieved context.

### Instructions

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.
Use the context to find relevant information and provide accurate answers. If the answer is not found in the context,
respond with "I don't know."
"""

### The user prompt template

In [ ]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

### Building the context

- The `context` is a formatted string with all the search results.
- Each document becomes a block with the section, question, and answer. This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. It's a small preprocessing step before we send the data to the LLM.

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

### Building the prompt

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

Let's try it:

In [ ]:
prompt = build_prompt(question, search_results)
print(prompt)

## The LLM

The last component of our RAG pipeline is the LLM. It takes the prompt we built and generates an answer.

### Sending the prompt to the LLM

We have the prompt from the previous section. We send it to the LLM:

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

### Exploring the response

The response is a Pydantic object. The answer is in `response.output` - a list of output items.
The first one is the message:

In [ ]:
print(response.output[0].content[0].text)
print("========================")
print(response.output_text)

The usage counts tell you how many tokens the request consumed:

In [ ]:
print(response.usage)

### Calculating the price

We're using [gpt-5.4-mini](https://developers.openai.com/api/docs/models/gpt-5.4-mini):

-   Input: $0.75 per million tokens
-   Output: $4.50 per million tokens

Let's calculate the cost of the request we just made:

In [ ]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

print(cost)

### Message history

Previously we sent only one string as input and got back a response. In practice, you typically send a message history - a list of messages where each message has a role.

Think of a ChatGPT conversation. It starts with a hidden system prompt that tells the LLM how to behave, one you never see. After that, your messages and the LLM's replies alternate. The LLM has no memory of its own, so it needs the full history passed in to continue the conversation.

We won't build a multi-turn chat here. But we still use this message format to separate our instructions from the user prompt.

We send two messages:

- `developer` - system-level instructions (how the LLM should behave)
    - OpenAI accepts both `developer` and `system` for the instruction role.
- `user` - the actual prompt with the question and context

In [ ]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [ ]:
print(response.output_text)
print("========================")
print(response.usage)

### The LLM function

We can now put this together into an updated `llm` function. It now takes both instructions and the user prompt:

In [ ]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

## Full RAG

With search, the prompt, and the LLM ready, we wire them together:

In [ ]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U
```

This approach is modular. You can swap the search backend, the prompt template, or the LLM model.

Nothing else needs to change. Later when we replace minsearch with sqlitesearch, only the `search` function changes.

In [ ]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

In [ ]:
answer = rag("How do I get a certificate?")
print(answer)